# Epic versus tragedy

In [14]:
import numpy as np
import pandas as pd

homer_df = pd.read_parquet("../parquet/homer_dtm.parquet")
tragedy_df = pd.read_parquet("../parquet/tragedy-with-years_dtm.parquet")

In [16]:
def log_likelihood(a: pd.Series, b: pd.Series, total_a: int, total_b: int) -> pd.Series:
    """Dunning G² for each lemma. Positive = skewed toward corpus A."""
    N = total_a + total_b
    expected_a = (a + b) * total_a / N
    expected_b = (a + b) * total_b / N

    def safe_term(obs, exp):
        return np.where(obs > 0, obs * np.log(obs / exp), 0)

    g2 = 2 * (safe_term(a, expected_a) + safe_term(b, expected_b))
    return pd.Series(np.where(a / total_a >= b / total_b, g2, -g2), index=a.index)

In [17]:
il = homer_df["iliad"]
od = homer_df["odyssey"]
homer_series = il + od
tragedy_series = tragedy_df.sum(axis=1)

homer_series, tragedy_series = homer_series.align(tragedy_series, fill_value=0)

total_homer = int(homer_series.sum())
total_tragedy = int(tragedy_series.sum())

loglikelihood_epic_tragedy = log_likelihood(homer_series, tragedy_series, total_homer, total_tragedy)

z_score_epic_tragedy = (
    (loglikelihood_epic_tragedy - loglikelihood_epic_tragedy.mean()) / loglikelihood_epic_tragedy.std()
)

/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [18]:
# Percentile rank of |G²| as heatmap intensity: robust to outliers,
# sign preserves Epic vs. Tragic direction.
heat_epic_tragedy = np.sign(loglikelihood_epic_tragedy) * loglikelihood_epic_tragedy.abs().rank(pct=True)
heat_epic_tragedy.sort_values()

lemma
λόγος    -0.999941
θνήσκω   -0.999793
εἷς      -0.999764
γῆ       -0.999705
ποός     -0.999557
            ...   
θυμός     0.999852
ναῦς      0.999882
Τρώς      0.999911
υἱός      0.999970
_         1.000000
Length: 33845, dtype: float64

In [ ]:
import altair as alt

from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

total_tokens = total_homer + total_tragedy
overall_freq = (homer_series + tragedy_series) / total_tokens

epic_tragedy_plot_df = pd.DataFrame({
    "lemma": loglikelihood_epic_tragedy.index,
    "log_likelihood": loglikelihood_epic_tragedy.values,
    "relative_frequency": overall_freq.values,
    "register": np.where(loglikelihood_epic_tragedy.values >= 0, "Epic", "Tragic"),
    "heat": heat_epic_tragedy.values,
})

# p-value from χ^2(df=1); use abs() because G^2 is signed here
epic_tragedy_plot_df["p_value"] = chi2.sf(epic_tragedy_plot_df["log_likelihood"].abs(), df=1)

# Benjamini-Hochberg FDR correction across all lemmata
_, epic_tragedy_plot_df["p_value_fdr"], _, _ = multipletests(epic_tragedy_plot_df["p_value"], method="fdr_bh")

epic_tragedy_plot_df = epic_tragedy_plot_df[epic_tragedy_plot_df["log_likelihood"].abs() >= 30]
print(f"{len(epic_tragedy_plot_df)} lemmata after filtering")
epic_tragedy_plot_df.sort_values("log_likelihood", ascending=False).tail(10)

33845 lemmata after filtering


,lemma,log_likelihood,relative_frequency,register,heat,p_value,p_value_fdr
31237,ἰώ,-364.523372,0.001284,Tragic,-0.999320,2.915074e-81,4.110861e-78
18634,τάλας,-368.262299,0.001380,Tragic,-0.999350,4.472474e-82,6.581342e-79
4849,δράω,-380.890272,0.001388,Tragic,-0.999380,7.964079e-85,1.225201e-81
18707,τέκνον,-408.687541,0.003053,Tragic,-0.999468,7.076460e-91,1.260541e-87
12952,οὗς,-439.472103,0.001548,Tragic,-0.999527,1.410383e-97,2.807908e-94
15230,ποός,-441.743276,0.001556,Tragic,-0.999557,4.519015e-98,9.559130e-95
3554,γῆ,-637.998356,0.002621,Tragic,-0.999705,9.104781e-141,2.801376e-137
5775,εἷς,-699.073973,0.004321,Tragic,-0.999764,4.754172e-154,1.787833e-150
6992,θνήσκω,-712.012941,0.002509,Tragic,-0.999793,7.302012e-157,3.089208e-153
10511,λόγος,-907.588404,0.003293,Tragic,-0.999941,2.198875e-199,2.480698e-195


In [20]:
alt.Chart(epic_tragedy_plot_df).mark_point(opacity=0.5, size=30).encode(
    x=alt.X("log_likelihood:Q", title="Log-likelihood (← Tragic | Epic →)"),
    y=alt.Y(
        "relative_frequency:Q",
        title="Overall relative frequency",
        scale=alt.Scale(type="log"),
    ),
    color=alt.Color(
        "register:N",
        scale=alt.Scale(domain=["Epic", "Tragic"], range=["tomato", "steelblue"]),
        legend=alt.Legend(title="More common in"),
    ),
    tooltip=["lemma:N", "log_likelihood:Q", "relative_frequency:Q"],
).properties(
    title="Epic vs. Tragic: lemma distinctiveness vs. overall frequency",
    width=650,
    height=450,
)

alt.Chart(...)

In [23]:
# epic_tragedy_plot_df.sort_values("log_likelihood", ascending=False).to_parquet("../parquet/epic_tragic_lemmata_loglikelihood.parquet")
epic_tragedy_plot_df.sort_values("log_likelihood", ascending=False).to_csv("../csv/epic_tragic.csv")

## Per-line "epicness" within tragedy

In [8]:
line_epicness_df = pd.read_parquet("../parquet/tragedy_line_epicness.parquet")

# Per-line heat: avg log-likelihood of that line's significant (|G²| >= 30) lemmata
line_epicness_df["heat"] = line_epicness_df["line_epicness"]

# Per-tragedy aggregate: mean of the per-line scores
tragedy_heat = (
    line_epicness_df.groupby(["dramatist", "title"])["line_epicness"]
    .mean()
    .sort_values()
)
tragedy_heat

dramatist  title               
Euripides  Suppliants             -76.442095
           Hippolytus             -70.407740
           Heracleidae            -69.539907
Sophocles  Electra                -69.093131
Euripides  Electra                -64.809881
           Orestes                -64.049772
Sophocles  OT                     -63.917001
Euripides  Alcestis               -63.757388
           Heracles               -63.405312
           Medea                  -63.176494
           Ion                    -61.822615
Sophocles  OC                     -60.451337
Euripides  Phoenician Women       -60.225409
           Andromache             -58.620874
           Helen                  -57.930766
           Hecuba                 -55.682263
Sophocles  Philoctetes            -55.578058
Euripides  IT                     -55.207495
Sophocles  Trachiniae             -53.824858
           Antigone               -52.884237
Euripides  IA                     -52.251054
           Trojan Women

In [11]:
line_epicness_df["line_urn"] = line_epicness_df["urn"] + ":" + line_epicness_df["n"].astype(str)

In [20]:
line_epicness_df[["line_urn", "heat"]].sort_values(by="heat", ascending=True)

,line_urn,heat
1498,urn:cts:greekLit:tlg0006.tlg013.perseus-grc2:1497,-1028.200819
19942,urn:cts:greekLit:tlg0006.tlg019.perseus-grc2:680,-1028.200819
4448,urn:cts:greekLit:tlg0006.tlg018.perseus-grc2:1459,-1028.200819
36281,urn:cts:greekLit:tlg0006.tlg001.perseus-grc2:674,-1028.200819
25747,urn:cts:greekLit:tlg0006.tlg009.perseus-grc2:1067,-1028.200819
...,...,...
43999,urn:cts:greekLit:tlg0011.tlg004.perseus-grc2:625,NaN
44001,urn:cts:greekLit:tlg0011.tlg004.perseus-grc2:626,NaN
44578,urn:cts:greekLit:tlg0011.tlg004.perseus-grc2:1220,NaN
44818,urn:cts:greekLit:tlg0011.tlg004.perseus-grc2:1471,NaN
